In [2]:
# 셀 2: 라이브러리 임포트 및 경로 설정
import pandas as pd
import os
import glob
from sklearn.model_selection import train_test_split
import shutil

# 사용자가 제공한 경로 설정
# Windows 경로의 '\'는 '\' 또는 '/'로 사용해야 합니다.
base_path = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer'
csv_path = os.path.join(base_path, 'csv')
image_path = os.path.join(base_path, 'wafer_images')

# CSV 파일이 있는 경로에서 모든 .csv 파일 목록 가져오기
csv_files = glob.glob(os.path.join(csv_path, '*.csv'))

# 파일을 제대로 찾았는지 확인
if not csv_files:
    print(f"경고: '{csv_path}' 경로에서 CSV 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
else:
    print(f"'{csv_path}' 경로에서 {len(csv_files)}개의 CSV 파일을 성공적으로 찾았습니다.")

# 모든 CSV 파일을 읽어 하나의 데이터프레임으로 결합
df_list = [pd.read_csv(file) for file in csv_files]
labels_df = pd.concat(df_list, ignore_index=True)

# 필요한 'ID'와 'Label' 컬럼만 선택하고 중복 데이터가 있다면 제거
labels_df = labels_df[['ID', 'Label']].drop_duplicates(subset=['ID']).reset_index(drop=True)

print(f"총 {len(csv_files)}개의 CSV 파일에서 {len(labels_df)}개의 고유 라벨 데이터를 불러왔습니다.")
print("\n--- 데이터 샘플 ---")
print(labels_df.head())
print("\n--- 라벨 분포 ---")
print(labels_df['Label'].value_counts())

'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\csv' 경로에서 20개의 CSV 파일을 성공적으로 찾았습니다.
총 20개의 CSV 파일에서 2000개의 고유 라벨 데이터를 불러왔습니다.

--- 데이터 샘플 ---
   ID   Label
0   1  Center
1   2  Center
2   3  Center
3   4  Center
4   5  Center

--- 라벨 분포 ---
Label
Center     260
Edge       260
Scratch    260
Crack      260
Loc        240
Donut      240
Random     240
Normal     240
Name: count, dtype: int64


In [3]:
# 셀 3: 클래스 목록 및 인덱스 생성
# 라벨을 알파벳 순으로 정렬하여 일관성 유지
class_names = sorted(labels_df['Label'].unique())
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for i, name in enumerate(class_names)}

# id를 키로, label을 값으로 하는 딕셔너리 생성
id_to_label_map = dict(zip(labels_df['ID'], labels_df['Label']))

print("--- 클래스 목록 (names) ---")
print(class_names)
print(f"\n총 클래스 개수: {len(class_names)}")

print("\n--- 클래스-인덱스 맵핑 (class_to_idx) ---")
print(class_to_idx)

--- 클래스 목록 (names) ---
['Center', 'Crack', 'Donut', 'Edge', 'Loc', 'Normal', 'Random', 'Scratch']

총 클래스 개수: 8

--- 클래스-인덱스 맵핑 (class_to_idx) ---
{'Center': 0, 'Crack': 1, 'Donut': 2, 'Edge': 3, 'Loc': 4, 'Normal': 5, 'Random': 6, 'Scratch': 7}


In [4]:
# 셀 4: YOLO 라벨 파일 생성
# 생성된 라벨 파일을 저장할 폴더 경로
label_output_path = os.path.join(base_path, 'wafer_labels')
os.makedirs(label_output_path, exist_ok=True)

# 이미지 파일 목록 가져오기
image_files = [f for f in os.listdir(image_path) if f.endswith('.png')]

# 각 이미지에 대해 라벨 파일 생성
for image_file in image_files:
    image_id = int(os.path.splitext(image_file)[0]) # '1.png' -> 1
    label = id_to_label_map.get(image_id)

    if label is not None:
        class_index = class_to_idx[label]
        yolo_label_content = f"{class_index} 0.5 0.5 1.0 1.0"
        
        label_file_path = os.path.join(label_output_path, f"{image_id}.txt")
        with open(label_file_path, 'w') as f:
            f.write(yolo_label_content)

print(f"'{label_output_path}' 폴더에 {len(image_files)}개의 YOLO 라벨 파일 생성을 완료했습니다.")

'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\wafer_labels' 폴더에 2000개의 YOLO 라벨 파일 생성을 완료했습니다.


In [5]:
# 셀 5: 데이터셋 폴더 구조 생성 및 파일 복사
# 전체 파일 목록 (ID 기준)
all_ids = labels_df['ID'].tolist()

# 학습용과 검증용 데이터 분리 (80% train, 20% valid)
train_ids, val_ids = train_test_split(all_ids, test_size=0.2, random_state=42)

# 데이터셋을 저장할 기본 폴더
dataset_path = os.path.join(base_path, 'yolo_dataset')

# 폴더 생성 함수
def create_dirs(ids, split_type):
    # split_type은 'train' 또는 'valid'
    img_dir = os.path.join(dataset_path, split_type, 'images')
    lbl_dir = os.path.join(dataset_path, split_type, 'labels')
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for id_ in ids:
        # 원본 파일 경로
        src_img_path = os.path.join(image_path, f"{id_}.png")
        src_lbl_path = os.path.join(label_output_path, f"{id_}.txt")

        # 대상 파일 경로
        dst_img_path = os.path.join(img_dir, f"{id_}.png")
        dst_lbl_path = os.path.join(lbl_dir, f"{id_}.txt")

        # 파일 복사
        if os.path.exists(src_img_path):
            shutil.copy(src_img_path, dst_img_path)
        if os.path.exists(src_lbl_path):
            shutil.copy(src_lbl_path, dst_lbl_path)

# 학습용 데이터 복사
create_dirs(train_ids, 'train')
# 검증용 데이터 복사
create_dirs(val_ids, 'valid')

print("YOLO 데이터셋 폴더 구조화 및 파일 복사를 완료했습니다.")
print(f"학습용 데이터: {len(train_ids)}개, 검증용 데이터: {len(val_ids)}개")
print(f"데이터셋 경로: {dataset_path}")

YOLO 데이터셋 폴더 구조화 및 파일 복사를 완료했습니다.
학습용 데이터: 1600개, 검증용 데이터: 400개
데이터셋 경로: C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset


In [6]:
# 셀 6: data.yaml 파일 생성
yaml_content = f"""
# 학습 및 검증 데이터 경로
train: {os.path.join(dataset_path, 'train', 'images')}
val: {os.path.join(dataset_path, 'valid', 'images')}

# 클래스 개수
nc: {len(class_names)}

# 클래스 이름 목록
names: {class_names}
"""

yaml_path = os.path.join(dataset_path, 'data.yaml')
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print(f"'{yaml_path}' 경로에 data.yaml 파일 생성을 완료했습니다.")
print("\n--- data.yaml 내용 ---")
print(yaml_content)

'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\data.yaml' 경로에 data.yaml 파일 생성을 완료했습니다.

--- data.yaml 내용 ---

# 학습 및 검증 데이터 경로
train: C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\train\images
val: C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\valid\images

# 클래스 개수
nc: 8

# 클래스 이름 목록
names: ['Center', 'Crack', 'Donut', 'Edge', 'Loc', 'Normal', 'Random', 'Scratch']



In [15]:
# 셀 7: YOLOv8 설치 (이미 설치했다면 이 셀은 건너뛰어도 됩니다)
!pip install ultralytics

You should consider upgrading via the 'C:\Users\Lee\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [18]:
!nvidia-smi

Thu Jun 19 05:13:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 576.40                 Driver Version: 576.40         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4080 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| 49%   53C    P3             43W /  320W |    2782MiB /  16376MiB |     15%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 셀 8: YOLOv8 모델 학습 실행(model=yolov8s.pt epochs=100 imgsz=640 device=0 batch=16)
!yolo train data="C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\data.yaml" model=yolov8s.pt epochs=100 imgsz=640 device=0 batch=16

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Lee\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.156  Python-3.10.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Lee\Downloads\\\4-1\\wafer\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_


  0%|          | 0.00/21.5M [00:00<?, ?B/s]
 26%|██▌       | 5.62M/21.5M [00:00<00:00, 58.4MB/s]
 53%|█████▎    | 11.4M/21.5M [00:00<00:00, 58.8MB/s]
 79%|███████▉  | 17.0M/21.5M [00:00<00:00, 56.8MB/s]
100%|██████████| 21.5M/21.5M [00:00<00:00, 56.2MB/s]

  0%|          | 0.00/755k [00:00<?, ?B/s]
100%|██████████| 755k/755k [00:00<00:00, 46.8MB/s]

  0%|          | 0.00/5.35M [00:00<?, ?B/s]
100%|██████████| 5.35M/5.35M [00:00<00:00, 58.2MB/s]

train: Scanning C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\train\labels...:   0%|          | 0/1600 [00:00<?, ?it/s]
train: Scanning C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\train\labels... 178 images, 0 backgrounds, 0 corrupt:  11%|█         | 178/1600 [00:00<00:00, 1763.72it/s]
train: Scanning C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\train\labels... 374 images, 0 backgrounds, 0 corrupt:  23%|██▎       | 374/1600 [00:00<00:00, 1877.10it/s]
train: Scanning C:\Users\Lee\Downloads\이승우\강남대

In [22]:
# 검증 셀 1
from ultralytics import YOLO
import os

print("YOLO 라이브러리 로드 완료.")

YOLO 라이브러리 로드 완료.


In [23]:
# 검증 셀 2

# 1. 훈련된 모델('best.pt')의 경로를 지정합니다.
#    .ipynb 파일의 위치를 기준으로 상대 경로를 사용하거나 절대 경로를 사용합니다.
model_path = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\best.pt' # 'runs/detect/train/weights/best.pt' 

# 2. 데이터셋 정보가 담긴 YAML 파일의 경로를 지정합니다.
data_yaml_path = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\data.yaml'

# 3. (선택) 검증에 사용할 이미지 크기를 지정합니다. (훈련 시와 동일한 크기 권장)
image_size = 640

# --- 경로 확인 ---
# 아래 코드는 경로가 올바른지 간단히 확인해줍니다.
if not os.path.exists(model_path):
    print(f"오류: 모델 파일을 찾을 수 없습니다. 경로를 확인하세요: {model_path}")
if not os.path.exists(data_yaml_path):
    print(f"오류: YAML 파일을 찾을 수 없습니다. 경로를 확인하세요: {data_yaml_path}")

print("경로 설정 완료.")

경로 설정 완료.


In [24]:
# 검증 셀 3

# YOLO 모델 로드
model = YOLO(model_path)

# 모델 검증(Validation) 수행
# model.val() 함수는 데이터셋에 대한 성능 지표(mAP, precision, recall 등)를 계산합니다.
print("모델 검증을 시작합니다...")
results = model.val(
    data=data_yaml_path,  # 데이터셋 설정 파일
    imgsz=image_size,     # 이미지 크기
    split='val',         # 'val' 데이터 세트를 사용
    batch=8,              # 배치 사이즈 (GPU 메모리에 따라 조절)
    save_json=True        # 결과를 JSON 파일로 저장
)
print("모델 검증 완료.")

모델 검증을 시작합니다...
Ultralytics 8.3.156  Python-3.10.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
Model summary (fused): 72 layers, 11,128,680 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 440.973.8 MB/s, size: 19.4 KB)


val: Scanning C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\yolo_dataset\valid\labels.cache... 400 images, 0 backgrounds, 0 corrupt: 100%|██████████| 400/400 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 50/50 [00:02<00:00, 23.95it/s]


                   all        400        400          1          1      0.995      0.995
                Center         51         51      0.999          1      0.995      0.995
                 Crack         42         42      0.999          1      0.995      0.995
                 Donut         56         56          1          1      0.995      0.995
                  Edge         52         52          1          1      0.995      0.995
                   Loc         44         44          1          1      0.995      0.995
                Normal         45         45          1          1      0.995      0.995
                Random         50         50          1          1      0.995      0.995
               Scratch         60         60      0.999          1      0.995      0.995
Speed: 0.1ms preprocess, 1.8ms inference, 0.0ms loss, 0.7ms postprocess per image
Saving runs\detect\val3\predictions.json...
Results saved to runs\detect\val3
모델 검증 완료.


In [25]:
# 검증 셀 4

# results 객체에서 주요 성능 지표를 추출하여 출력
print("--- 검증 결과 요약 ---")
print(f"mAP@50-95: {results.box.map:.4f}")  # mAP at IoU 0.50-0.95
print(f"mAP@50:    {results.box.map50:.4f}")   # mAP at IoU 0.50
print(f"mAP@75:    {results.box.map75:.4f}")   # mAP at IoU 0.75
print("--------------------")

# results 객체의 전체 내용이 궁금하다면 아래 주석을 풀고 실행해보세요.
# print(results)

--- 검증 결과 요약 ---
mAP@50-95: 0.9950
mAP@50:    0.9950
mAP@75:    0.9950
--------------------


In [ ]:
# best.pt 모델로 새로운 이미지를 예측(추론)하는 명령어
# source="..." 부분에 테스트할 이미지 파일 경로
# !yolo predict model=runs/detect/train/weights/best.pt source="C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\test3"
!yolo predict model=best.pt source="C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\test\test1"

Ultralytics 8.3.156  Python-3.10.0 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4080 SUPER, 16376MiB)
Model summary (fused): 72 layers, 11,128,680 parameters, 0 gradients, 28.5 GFLOPs

image 1/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1205.png: 640x640 1 Center, 4.0ms
image 2/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1208.png: 640x640 1 Center, 3.8ms
image 3/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1219.png: 640x640 1 Edge, 3.4ms
image 4/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1286.png: 640x640 1 Random, 3.2ms
image 5/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1296.png: 640x640 1 Edge, 3.8ms
image 6/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1385.png: 640x640 1 Random, 3.1ms
image 7/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1408.png: 640x640 1 Center, 3.3ms
image 8/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1443.png: 640x640 1 Crack, 3.5ms
image 9/25 C:\Users\Lee\Downloads\\\4-1\\wafer\test1\1471.png: 640x640 1 Edge, 3.0ms
image 10/25 C:\Users\Lee\Downloads\\\4-

In [ ]:
# YOLO 모델로 이미지 분류 후 유형별로 폴더 자동 정리
import os
import shutil
import glob
from ultralytics import YOLO
from tqdm import tqdm # 진행 상황을 시각적으로 보여주는 라이브러리
from datetime import datetime

# 1. 사용자 환경에 맞게 경로 수정
# 학습된 나의 best.pt 모델 파일 경로
MODEL_PATH = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\best.pt' # 'runs/detect/train/weights/best.pt'

# 분류하고 싶은 새로운 이미지들이 들어있는 폴더 경로
SOURCE_IMAGES_PATH = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\test\test1'

# 분류된 이미지들을 저장할 상위 폴더 경로, 이 폴더에 'Center', 'Donut' 등 유형 폴더가 자동으로 생성
# OUTPUT_BASE_PATH = 'runs/detect/results'

# 현재 시간을 '년월일_시분초' 형태의 문자열로 만듦 (예:'20250101_123456')
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S") 
# 'results_' 뒤에 타임스탬프를 붙여 폴더 경로를 완성
OUTPUT_BASE_PATH = os.path.join('runs/detect', f'results({timestamp})')
print(f"'{OUTPUT_BASE_PATH}' 폴더에 저장됩니다.")

# 2. 모델 로드
try:
    model = YOLO(MODEL_PATH)
    print(f"'{MODEL_PATH}' 모델을 성공적으로 불러왔습니다.")
    print("클래스 목록:", model.names)
except Exception as e:
    print(f"모델 로딩 중 오류 발생: {e}")
    # 모델 로딩에 실패하면 여기서 중단
    model = None

if model:
    # 3. 분류할 이미지 파일 목록 가져오기
    # .png, .jpg, .jpeg 등 다양한 확장자 지원
    image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.gif']
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(SOURCE_IMAGES_PATH, ext)))
    
    if not image_files:
        print(f"경고: '{SOURCE_IMAGES_PATH}' 폴더에 분류할 이미지가 없습니다. 경로를 확인해주세요.")
    else:
        print(f"총 {len(image_files)}개의 이미지를 분류합니다...")

        # 4. 분류 작업 시작
        os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)

        # tqdm을 사용하여 진행률 표시
        for image_path in tqdm(image_files, desc="분류 진행률"):
            try:
                # 모델 예측 실행 (상세 로그는 끄기)
                results = model.predict(source=image_path, verbose=False)
                result = results[0]

                # 예측된 결과가 있을 경우
                if result.boxes:
                    # 가장 확률이 높은 예측 결과의 클래스 ID를 가져옴
                    top_class_id = int(result.boxes.cls[0])
                    # 클래스 ID를 실제 클래스 이름으로 변환
                    predicted_class_name = model.names[top_class_id]
                else:
                    # 모델이 아무것도 예측하지 못한 경우
                    predicted_class_name = "Unclassified" # '미분류' 폴더로 이동

                # 5. 결과에 따라 폴더 생성 및 파일 복사
                destination_folder = os.path.join(OUTPUT_BASE_PATH, predicted_class_name)
                os.makedirs(destination_folder, exist_ok=True)
                
                # 원본 파일을 해당 폴더로 '복사'합니다. (이동을 원하면 shutil.move)
                shutil.copy(image_path, destination_folder)

            except Exception as e:
                print(f"'{os.path.basename(image_path)}' 처리 중 오류 발생: {e}")
        
        print("\n모든 이미지 분류 및 폴더 정리를 완료했습니다!")
        print(f"결과는 '{OUTPUT_BASE_PATH}' 폴더에서 확인하실 수 있습니다.")

'runs/detect\results(20250620_042502)' 폴더에 저장됩니다.
'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\best.pt' 모델을 성공적으로 불러왔습니다.
클래스 목록: {0: 'Center', 1: 'Crack', 2: 'Donut', 3: 'Edge', 4: 'Loc', 5: 'Normal', 6: 'Random', 7: 'Scratch'}
총 25개의 이미지를 분류합니다...


분류 진행률: 100%|██████████| 25/25 [00:01<00:00, 21.88it/s]


모든 이미지 분류 및 폴더 정리를 완료했습니다!
결과는 'runs/detect\results(20250620_042502)' 폴더에서 확인하실 수 있습니다.


In [34]:
# YOLO 모델로 이미지 분류 후 유형별로 폴더 자동 정리 + CSV 라벨 자동 기입 
import os
import shutil
import glob
import pandas as pd # pandas 라이브러리 추가
from ultralytics import YOLO
from tqdm import tqdm
from datetime import datetime

# 1. 사용자 환경에 맞게 경로 수정
# 학습된 나의 best.pt 모델 파일 경로
MODEL_PATH = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\best.pt' # 'runs/detect/train/weights/best.pt'

# 분류하고 싶은 새로운 이미지들이 들어있는 폴더 경로
SOURCE_IMAGES_PATH = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\test\test1'

# 라벨을 채워넣을 CSV 파일 경로
CSV_PATH = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\test\test1\wafer_batch_1.csv'

# 현재 시간을 '년월일_시분초' 형태의 문자열로 만듦 (예:'20250101_123456')
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# 'results_' 뒤에 타임스탬프를 붙여 폴더 경로를 완성
OUTPUT_BASE_PATH = os.path.join('runs/detect', f'results({timestamp})')
print(f"'{OUTPUT_BASE_PATH}' 폴더에 분류된 이미지와 CSV 파일이 저장됩니다.")

# 2. 모델 로드
try:
    model = YOLO(MODEL_PATH)
    print(f"'{MODEL_PATH}' 모델을 성공적으로 불러왔습니다.")
    print("클래스 목록:", model.names)
except Exception as e:
    print(f"모델 로딩 중 오류 발생: {e}")
    # 모델 로딩에 실패하면 여기서 중단
    model = None

# 3. CSV 파일 로드
try:
    df = pd.read_csv(CSV_PATH)
    # ID를 인덱스로 설정하여 빠른 조회를 위함
    # 'ID' 컬럼이 없는 경우를 대비한 오류 처리
    if 'ID' in df.columns:
        df.set_index('ID', inplace=True)
        print(f"'{CSV_PATH}' 파일을 성공적으로 불러왔습니다.")
    else:
        df = None
        print(f"오류: '{CSV_PATH}' 파일에 'ID' 컬럼이 없습니다.")
except FileNotFoundError:
    df = None
    print(f"오류: '{CSV_PATH}' 파일을 찾을 수 없습니다.")

# 모델과 데이터프레임이 모두 성공적으로 로드된 경우에만 실행
if model and df is not None:
    # 4. 분류할 이미지 파일 목록 가져오기
    # .png, .jpg, .jpeg 등 다양한 확장자 지원
    image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.gif']
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(SOURCE_IMAGES_PATH, ext)))
    
    if not image_files:
        print(f"경고: '{SOURCE_IMAGES_PATH}' 폴더에 분류할 이미지가 없습니다. 경로를 확인해주세요.")
    else:
        print(f"총 {len(image_files)}개의 이미지를 분류하고 CSV에 라벨을 기입합니다...")

        # 5. 분류 작업 시작
        os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)

        # tqdm을 사용하여 진행률 표시
        for image_path in tqdm(image_files, desc="분류 진행률"):
            try:
                # 모델 예측 실행 (상세 로그는 끄기)
                results = model.predict(source=image_path, verbose=False)
                result = results[0]
                
                # 이미지 파일 이름에서 ID 추출 (예: '123.png' -> 123)
                image_filename = os.path.basename(image_path)
                image_id = int(os.path.splitext(image_filename)[0])

                # 예측된 결과가 있을 경우
                if result.boxes:
                    # 가장 확률이 높은 예측 결과의 클래스 ID를 가져옴
                    top_class_id = int(result.boxes.cls[0])
                    # 클래스 ID를 실제 클래스 이름으로 변환
                    predicted_class_name = model.names[top_class_id]
                else:
                    # 모델이 아무것도 예측하지 못한 경우
                    predicted_class_name = "Unclassified" # '미분류' 폴더로 이동

                # csv 업데이트
                # 추출한 image_id가 DataFrame의 인덱스(ID)에 존재하는지 확인
                if image_id in df.index:
                    # 'Label' 컬럼에 예측된 클래스 이름 기입
                    df.loc[image_id, 'Label'] = predicted_class_name
                else:
                    print(f"\n경고: ID {image_id}가 CSV 파일에 없어 라벨을 기입할 수 없습니다.")

                # 6. 결과에 따라 폴더 생성 및 파일 복사
                destination_folder = os.path.join(OUTPUT_BASE_PATH, predicted_class_name)
                os.makedirs(destination_folder, exist_ok=True)

                # 원본 파일을 해당 폴더로 '복사'합니다. (이동을 원하면 shutil.move)
                shutil.copy(image_path, destination_folder)

            except Exception as e:
                print(f"'{image_filename}' 처리 중 오류 발생: {e}")
        
        # 7. 최종 CSV 저장 로직
        try:
            # 인덱스(ID)를 다시 컬럼으로 복원
            df.reset_index(inplace=True)
            # 새로운 CSV 파일 이름 설정
            output_csv_name = f"{os.path.splitext(os.path.basename(CSV_PATH))[0]}_labeled.csv"
            output_csv_path = os.path.join(OUTPUT_BASE_PATH, output_csv_name)
            
            # 새 CSV 파일로 저장 (기존 파일 덮어쓰기 방지)
            df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
            
            print("\n모든 이미지 분류 및 폴더 정리를 완료했습니다!")
            print(f"결과는 '{OUTPUT_BASE_PATH}' 폴더에서 확인하실 수 있습니다.")
            print(f"라벨이 추가된 CSV 파일: '{output_csv_path}'")

        except Exception as e:
            print(f"\nCSV 파일 저장 중 오류 발생: {e}")

'runs/detect\results(20250620_053456)' 폴더에 분류된 이미지와 CSV 파일이 저장됩니다.
'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\best.pt' 모델을 성공적으로 불러왔습니다.
클래스 목록: {0: 'Center', 1: 'Crack', 2: 'Donut', 3: 'Edge', 4: 'Loc', 5: 'Normal', 6: 'Random', 7: 'Scratch'}
'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\test\test1\wafer_batch_1.csv' 파일을 성공적으로 불러왔습니다.
총 25개의 이미지를 분류하고 CSV에 라벨을 기입합니다...


분류 진행률:   0%|          | 0/25 [00:00<?, ?it/s]C:\Users\Lee\AppData\Local\Temp\ipykernel_9176\3031838726.py:93: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Center' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[image_id, 'Label'] = predicted_class_name
분류 진행률: 100%|██████████| 25/25 [00:01<00:00, 22.13it/s]


모든 이미지 분류 및 폴더 정리를 완료했습니다!
결과는 'runs/detect\results(20250620_053456)' 폴더에서 확인하실 수 있습니다.
라벨이 추가된 CSV 파일: 'runs/detect\results(20250620_053456)\wafer_batch_1_labeled.csv'


In [6]:
# YOLO 모델로 이미지 분류 후 유형별로 폴더 자동 정리 + CSV 라벨 자동 기입 + 이미지&csv 경로 통일
import os
import shutil
import glob
import pandas as pd # pandas 라이브러리 추가
from ultralytics import YOLO
from tqdm import tqdm
from datetime import datetime

# 1. 사용자 환경에 맞게 경로 수정
# 학습된 나의 best.pt 모델 파일 경로
MODEL_PATH = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\best.pt' # 'runs/detect/train/weights/best.pt'

# 분류하고 싶은 새로운 이미지들과 csv 파일이 들어있는 폴더 경로
SOURCE_PATH = r'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\test\test5'

# 현재 시간을 '년월일_시분초' 형태의 문자열로 만듦 (예:'20250101_123456')
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# 'results_' 뒤에 타임스탬프를 붙여 폴더 경로를 완성
OUTPUT_BASE_PATH = os.path.join('runs/detect', f'results({timestamp})')
print(f"'{OUTPUT_BASE_PATH}' 폴더에 분류된 이미지와 CSV 파일이 저장됩니다.")

# 2. 모델 로드
try:
    model = YOLO(MODEL_PATH)
    print(f"'{MODEL_PATH}' 모델을 성공적으로 불러왔습니다.")
    print("클래스 목록:", model.names)
except Exception as e:
    print(f"모델 로딩 중 오류 발생: {e}")
    # 모델 로딩에 실패하면 여기서 중단
    model = None

# 3. CSV 파일 로드
csv_path_to_process = None
df = None
try:
    # 소스 폴더에서 .csv 파일 검색
    found_csv_files = glob.glob(os.path.join(SOURCE_PATH, '*.csv'))
    
    if len(found_csv_files) == 1:
        # CSV 파일이 정확히 하나일 때
        csv_path_to_process = found_csv_files[0]
        df = pd.read_csv(csv_path_to_process)
        if 'ID' in df.columns:
            df.set_index('ID', inplace=True)
            print(f"CSV 파일 '{os.path.basename(csv_path_to_process)}'을(를) 불러와 라벨링을 준비합니다.")
        else:
            df = None
            print(f"경고: '{os.path.basename(csv_path_to_process)}' 파일에 'ID' 컬럼이 없어 CSV 라벨링을 건너뜁니다.")
    elif len(found_csv_files) > 1:
        # CSV 파일이 두 개 이상일 때
        print(f"경고: 소스 폴더에 CSV 파일이 여러 개({len(found_csv_files)}개) 있습니다. CSV 라벨링을 건너뜁니다.")
    else:
        # CSV 파일이 없을 때
        print("정보: 소스 폴더에 CSV 파일이 없어 CSV 라벨링을 건너뜁니다.")
        
except Exception as e:
    df = None
    print(f"CSV 파일 처리 중 오류 발생: {e}")


# 모델과 데이터프레임이 모두 성공적으로 로드된 경우에만 실행
if model:
    # 4. 분류할 이미지 파일 목록 가져오기
    # .png, .jpg, .jpeg 등 다양한 확장자 지원
    image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.gif']
    image_files = []
    for ext in image_extensions:
        image_files.extend(glob.glob(os.path.join(SOURCE_PATH, ext)))
    
    if not image_files:
        print(f"경고: '{SOURCE_PATH}' 폴더에 분류할 이미지가 없습니다. 경로를 확인해주세요.")
    else:
        print(f"총 {len(image_files)}개의 이미지를 분류하고 CSV에 라벨을 기입합니다...")

        # 5. 분류 작업 시작
        os.makedirs(OUTPUT_BASE_PATH, exist_ok=True)

        # tqdm을 사용하여 진행률 표시
        for image_path in tqdm(image_files, desc="분류 진행률"):
            try:
                # 모델 예측 실행 (상세 로그는 끄기)
                results = model.predict(source=image_path, verbose=False)
                result = results[0]
                
                # 이미지 파일 이름에서 ID 추출 (예: '123.png' -> 123)
                image_filename = os.path.basename(image_path)
                image_id = int(os.path.splitext(image_filename)[0])

                # 예측된 결과가 있을 경우
                if result.boxes:
                    # 가장 확률이 높은 예측 결과의 클래스 ID를 가져옴
                    top_class_id = int(result.boxes.cls[0])
                    # 클래스 ID를 실제 클래스 이름으로 변환
                    predicted_class_name = model.names[top_class_id]
                else:
                    # 모델이 아무것도 예측하지 못한 경우
                    predicted_class_name = "Unclassified" # '미분류' 폴더로 이동

                # csv 업데이트
                # df가 성공적으로 로드되었을 때만 라벨링 수행
                if df is not None:
                    if image_id in df.index:
                        df.loc[image_id, 'Label'] = predicted_class_name
                    else:
                        print(f"\n경고: ID {image_id}가 CSV 파일에 없어 라벨을 기입할 수 없습니다.")

                # 6. 결과에 따라 폴더 생성 및 파일 복사
                destination_folder = os.path.join(OUTPUT_BASE_PATH, predicted_class_name)
                os.makedirs(destination_folder, exist_ok=True)

                # 원본 파일을 해당 폴더로 '복사'합니다. (이동을 원하면 shutil.move)
                shutil.copy(image_path, destination_folder)

            except Exception as e:
                print(f"'{image_filename}' 처리 중 오류 발생: {e}")
        
        # 7. 최종 CSV 저장 로직(df가 성공적으로 로드되었을 때만 CSV 저장)
        if df is not None:
            try:
                # 인덱스(ID)를 다시 컬럼으로 복원
                df.reset_index(inplace=True)
                # 새로운 CSV 파일 이름 설정
                output_csv_name = f"{os.path.splitext(os.path.basename(csv_path_to_process))[0]}_labeled.csv"
                output_csv_path = os.path.join(OUTPUT_BASE_PATH, output_csv_name)

                # 새 CSV 파일로 저장 (기존 파일 덮어쓰기 방지)
                df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

                print(f"\n라벨이 추가된 CSV 파일: '{output_csv_path}'")
            except Exception as e:
                print(f"\nCSV 파일 저장 중 오류 발생: {e}")

        print("모든 이미지 분류 및 폴더 정리를 완료했습니다!")
        print(f"결과는 '{OUTPUT_BASE_PATH}' 폴더에서 확인하실 수 있습니다.")

'runs/detect\results(20250620_064525)' 폴더에 분류된 이미지와 CSV 파일이 저장됩니다.
'C:\Users\Lee\Downloads\이승우\강남대\4-1\캡스톤디자인\wafer\best.pt' 모델을 성공적으로 불러왔습니다.
클래스 목록: {0: 'Center', 1: 'Crack', 2: 'Donut', 3: 'Edge', 4: 'Loc', 5: 'Normal', 6: 'Random', 7: 'Scratch'}
CSV 파일 'wafer_batch_5.csv'을(를) 불러와 라벨링을 준비합니다.
총 25개의 이미지를 분류하고 CSV에 라벨을 기입합니다...


분류 진행률:   0%|          | 0/25 [00:00<?, ?it/s]C:\Users\Lee\AppData\Local\Temp\ipykernel_9996\2321921143.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Crack' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[image_id, 'Label'] = predicted_class_name
분류 진행률: 100%|██████████| 25/25 [00:01<00:00, 22.23it/s]


라벨이 추가된 CSV 파일: 'runs/detect\results(20250620_064525)\wafer_batch_5_labeled.csv'
모든 이미지 분류 및 폴더 정리를 완료했습니다!
결과는 'runs/detect\results(20250620_064525)' 폴더에서 확인하실 수 있습니다.


In [ ]:
# YOLO 모델로 이미지 분류 후 유형별로 폴더 자동 정리 PyQt 버전
import sys
import os
import shutil
import glob
from PyQt5.QtWidgets import (QApplication, QWidget, QVBoxLayout, QHBoxLayout, QGroupBox,
                             QPushButton, QLineEdit, QFileDialog, QLabel, 
                             QProgressBar, QPlainTextEdit, QRadioButton)
from PyQt5.QtCore import QThread, pyqtSignal
from ultralytics import YOLO
from datetime import datetime

# 모델 경로
MODEL_PATH = 'runs/detect/train/weights/best.pt'

# 긴 작업(YOLO 예측)을 처리할 별도의 스레드
class Worker(QThread):
    # 시그널 정의: 진행률 업데이트, 로그 메시지, 작업 완료
    progress = pyqtSignal(int)
    log = pyqtSignal(str)
    finished = pyqtSignal(str)

    def __init__(self, model_path, source_path, output_path):
        super().__init__()
        self.model_path = model_path
        self.source_path = source_path
        self.output_path = output_path
        self.is_running = True

    def run(self):
        try:
            if not os.path.exists(self.model_path):
                self.log.emit(f"오류: 모델 파일을 찾을 수 없습니다. 경로를 확인해주세요: {self.model_path}")
                self.finished.emit("")
                return
            
            self.log.emit(f"'{os.path.basename(self.model_path)}' 모델 로딩 중...")
            model = YOLO(self.model_path)
            self.log.emit(f"모델 로딩 성공. 클래스: {list(model.names.values())}")

            if os.path.isdir(self.source_path):
                image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.gif']
                image_files = []
                for ext in image_extensions:
                    image_files.extend(glob.glob(os.path.join(self.source_path, ext)))
                if not image_files:
                    self.log.emit(f"오류: '{self.source_path}' 폴더에 이미지가 없습니다.")
                    self.finished.emit()
                    return
            
            elif os.path.isfile(self.source_path):
                image_files = [self.source_path]

            else:
                self.log.emit(f"오류: 유효하지 않은 경로입니다: '{self.source_path}'")
                self.finished.emit()
                return
            
            self.log.emit(f"총 {len(image_files)}개의 이미지 분류를 시작합니다.")
            os.makedirs(self.output_path, exist_ok=True)
            
            for i, image_path in enumerate(image_files):
                if not self.is_running:
                    break

                results = model.predict(source=image_path, verbose=False)
                result = results[0]

                if result.boxes:
                    top_class_id = int(result.boxes.cls[0])
                    predicted_class_name = model.names[top_class_id]
                else:
                    predicted_class_name = "Unclassified"

                destination_folder = os.path.join(self.output_path, predicted_class_name)
                os.makedirs(destination_folder, exist_ok=True)
                shutil.copy(image_path, destination_folder)
                
                self.log.emit(f"'{os.path.basename(image_path)}' -> '{predicted_class_name}' 폴더로 복사 완료")
                self.progress.emit(int((i + 1) / len(image_files) * 100))

            if self.is_running:
                self.log.emit("모든 작업이 완료되었습니다!")
            else:
                self.log.emit("작업이 중단되었습니다.")

        except Exception as e:
            self.log.emit(f"오류 발생: {e}")
        finally:
            self.finished.emit(self.output_path)

    def stop(self):
        self.is_running = False

# 메인 윈도우
class ClassifierApp(QWidget):
    def __init__(self):
        super().__init__()
        self.initUI()
        self.worker = None

    def initUI(self):
        self.setWindowTitle('Wafer Map 분류기')
        
        # 레이아웃 설정
        vbox = QVBoxLayout()
        self.setLayout(vbox)

        # 1. 폴더/파일 선택 옵션 UI 추가 ---
        group_box = QGroupBox("이미지 폴더/파일 선택")
        self.radio_layout = QHBoxLayout()
        self.radio_folder = QRadioButton("폴더")
        self.radio_file = QRadioButton("파일")
        self.radio_folder.setChecked(True) # 기본값으로 폴더 선택
        self.radio_layout.addWidget(self.radio_folder)
        self.radio_layout.addWidget(self.radio_file)
        group_box.setLayout(self.radio_layout)
        vbox.addWidget(group_box)

        # 2. 소스 폴더 선택
        hbox2 = QHBoxLayout()
        self.source_label = QLabel('이미지:')
        self.source_path_edit = QLineEdit()
        self.source_path_edit.setReadOnly(True)
        self.source_btn = QPushButton('선택...')
        self.source_btn.clicked.connect(self.select_source)
        hbox2.addWidget(self.source_label)
        hbox2.addWidget(self.source_path_edit)
        hbox2.addWidget(self.source_btn)
        vbox.addLayout(hbox2)

        # 3. 실행 버튼
        self.run_btn = QPushButton('분류 시작')
        self.run_btn.clicked.connect(self.start_classification)
        vbox.addWidget(self.run_btn)

        # 4. 진행률 바
        self.progress_bar = QProgressBar()
        vbox.addWidget(self.progress_bar)

        # 5. 로그 창
        self.log_edit = QPlainTextEdit()
        self.log_edit.setReadOnly(True)
        vbox.addWidget(self.log_edit)

        self.setGeometry(300, 300, 600, 400)
        self.show()

    def select_source(self):
        if self.radio_folder.isChecked():
            path = QFileDialog.getExistingDirectory(self, '이미지 폴더 선택')
        else: # self.radio_file.isChecked()
            path, _ = QFileDialog.getOpenFileName(self, '이미지 파일 선택', '', 'Image Files (*.png *.jpg *.jpeg *.bmp *.gif)')
        
        if path:
            self.source_path_edit.setText(path)

    def start_classification(self):
        model_path = MODEL_PATH
        source_path = self.source_path_edit.text()

        if not source_path:
            self.log_edit.appendPlainText("오류: '입력 경로'를 선택해주세요.")
            return

        # 타임스탬프 생성
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join('runs/detect', f'results({timestamp})')

        self.run_btn.setEnabled(False)
        self.progress_bar.setValue(0)
        self.log_edit.clear()

        self.worker = Worker(model_path, source_path, output_path)
        self.worker.progress.connect(self.progress_bar.setValue)
        self.worker.log.connect(self.log_edit.appendPlainText)
        self.worker.finished.connect(self.on_finished)
        self.worker.start()

    def on_finished(self, result_path):
        self.run_btn.setEnabled(True)
        self.worker = None # 다음 실행을 위해 worker 참조를 초기화

        # 전달받은 경로가 비어있지 않은 경우 (오류 없이 정상 종료된 경우) 메시지 표시
        if result_path:
            self.log_edit.appendPlainText(f"\n결과가 '{result_path}' 폴더에 저장되었습니다.")

    def closeEvent(self, event):
        if self.worker is not None and self.worker.isRunning():
            print("Closing... Stopping worker thread. Please wait.") # 터미널에 메시지 출력
            self.log_edit.appendPlainText("프로그램 종료 중... 작업 스레드가 멈추기를 기다립니다.")
            self.worker.stop()
            self.worker.wait() # 스레드가 완전히 끝날 때까지 기다림
            print("Worker thread finished. Exiting.")
        event.accept()

if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = ClassifierApp()
    sys.exit(app.exec_())

SystemExit: 0

C:\Users\Lee\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
# YOLO 모델로 이미지 분류 후 유형별로 폴더 자동 정리 PyQt 버전 + csv 라벨 자동 기입
import sys
import os
import shutil
import glob
import pandas as pd
from PyQt5.QtWidgets import (QApplication, QWidget, QVBoxLayout, QHBoxLayout, QGroupBox,
                             QPushButton, QLineEdit, QFileDialog, QLabel, 
                             QProgressBar, QPlainTextEdit, QRadioButton)
from PyQt5.QtCore import QThread, pyqtSignal
from ultralytics import YOLO
from datetime import datetime

# 모델 경로
MODEL_PATH = 'runs/detect/train/weights/best.pt'

# 긴 작업(YOLO 예측)을 처리할 별도의 스레드
class Worker(QThread):
    # 시그널 정의: 진행률 업데이트, 로그 메시지, 작업 완료
    progress = pyqtSignal(int)
    log = pyqtSignal(str)
    finished = pyqtSignal(str)

    def __init__(self, model_path, source_path, output_path):
        super().__init__()
        self.model_path = model_path
        self.source_path = source_path
        self.output_path = output_path
        self.is_running = True

    def run(self):
        try:
            # 1. 모델 로드
            if not os.path.exists(self.model_path):
                self.log.emit(f"오류: 모델 파일을 찾을 수 없습니다. 경로를 확인해주세요: {self.model_path}")
                self.finished.emit("")
                return
            
            self.log.emit(f"'{os.path.basename(self.model_path)}' 모델 로딩 중...")
            model = YOLO(self.model_path)
            self.log.emit(f"모델 로딩 성공. 클래스: {list(model.names.values())}")

            # 2. CSV 파일 자동 탐지 및 로드
            csv_path_to_process = None
            df = None
            source_dir = self.source_path if os.path.isdir(self.source_path) else os.path.dirname(self.source_path)
            
            found_csv_files = glob.glob(os.path.join(source_dir, '*.csv'))
            if len(found_csv_files) == 1:
                csv_path_to_process = found_csv_files[0]
                df = pd.read_csv(csv_path_to_process)
                if 'ID' in df.columns:
                    df.set_index('ID', inplace=True)
                    self.log.emit(f"CSV 파일 '{os.path.basename(csv_path_to_process)}'을(를) 불러와 라벨링을 준비합니다.")
                else:
                    df = None
                    self.log.emit(f"경고: '{os.path.basename(csv_path_to_process)}' 파일에 'ID' 컬럼이 없어 CSV 라벨링을 건너뜁니다.")
            elif len(found_csv_files) > 1:
                self.log.emit(f"경고: 소스 폴더에 CSV 파일이 여러 개({len(found_csv_files)}개) 있습니다. CSV 라벨링을 건너뜁니다.")
            else:
                self.log.emit("정보: 소스 폴더에 CSV 파일이 없어 CSV 라벨링을 건너뜁니다.")

            # 3. 이미지 파일 목록 가져오기
            if os.path.isdir(self.source_path):
                image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.gif']
                image_files = []
                for ext in image_extensions:
                    image_files.extend(glob.glob(os.path.join(self.source_path, ext)))
                if not image_files:
                    self.log.emit(f"오류: '{self.source_path}' 폴더에 이미지가 없습니다.")
                    self.finished.emit("")
                    return
            
            elif os.path.isfile(self.source_path):
                image_files = [self.source_path]

            else:
                self.log.emit(f"오류: 유효하지 않은 경로입니다: '{self.source_path}'")
                self.finished.emit()
                return
            
            self.log.emit(f"총 {len(image_files)}개의 이미지 분류를 시작합니다.")
            os.makedirs(self.output_path, exist_ok=True)

            # 4. 이미지 분류, 파일 복사, CSV 업데이트 루프
            for i, image_path in enumerate(image_files):
                if not self.is_running:
                    break
                
                results = model.predict(source=image_path, verbose=False)
                result = results[0]
                image_filename = os.path.basename(image_path)
                image_id = int(os.path.splitext(image_filename)[0])

                if result.boxes:
                    top_class_id = int(result.boxes.cls[0])
                    predicted_class_name = model.names[top_class_id]
                else:
                    predicted_class_name = "Unclassified"

                # df가 있을 경우에만 라벨링 수행
                if df is not None:
                    if image_id in df.index:
                        df.loc[image_id, 'Label'] = predicted_class_name
                    else:
                        self.log.emit(f"경고: ID {image_id}가 CSV 파일에 없어 라벨을 기입할 수 없습니다.")

                # 파일 복사
                destination_folder = os.path.join(self.output_path, predicted_class_name)
                os.makedirs(destination_folder, exist_ok=True)
                shutil.copy(image_path, destination_folder)
                
                self.log.emit(f"'{image_filename}' -> '{predicted_class_name}'")
                self.progress.emit(int((i + 1) / len(image_files) * 100))

            # 5. 최종 CSV 파일 저장
            if df is not None:
                try:
                    df.reset_index(inplace=True)
                    output_csv_name = f"{os.path.splitext(os.path.basename(csv_path_to_process))[0]}_labeled.csv"
                    output_csv_path = os.path.join(self.output_path, output_csv_name)
                    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
                    self.log.emit(f"\n라벨이 추가된 CSV 파일: '{output_csv_path}'")
                except Exception as e:
                    self.log.emit(f"\nCSV 파일 저장 중 오류 발생: {e}")

            if self.is_running:
                self.log.emit("\n모든 작업이 완료되었습니다!")
            else:
                self.log.emit("\n작업이 중단되었습니다.")
        except Exception as e:
            self.log.emit(f"오류 발생: {e}")
        finally:
            self.finished.emit(self.output_path)

    def stop(self):
        self.is_running = False

# 메인 윈도우
class ClassifierApp(QWidget):
    def __init__(self):
        super().__init__()
        self.initUI()
        self.worker = None

    def initUI(self):
        self.setWindowTitle('Wafer Map 분류기')
        
        # 레이아웃 설정
        vbox = QVBoxLayout()
        self.setLayout(vbox)

        # 1. 폴더/파일 선택 옵션 UI 추가 ---
        group_box = QGroupBox("이미지 폴더/파일 선택")
        self.radio_layout = QHBoxLayout()
        self.radio_folder = QRadioButton("폴더")
        self.radio_file = QRadioButton("파일")
        self.radio_folder.setChecked(True) # 기본값으로 폴더 선택
        self.radio_layout.addWidget(self.radio_folder)
        self.radio_layout.addWidget(self.radio_file)
        group_box.setLayout(self.radio_layout)
        vbox.addWidget(group_box)

        # 2. 소스 폴더 선택
        hbox2 = QHBoxLayout()
        self.source_label = QLabel('입력 경로:')
        self.source_path_edit = QLineEdit()
        self.source_path_edit.setReadOnly(True)
        self.source_btn = QPushButton('경로 선택...')
        self.source_btn.clicked.connect(self.select_source)
        hbox2.addWidget(self.source_label)
        hbox2.addWidget(self.source_path_edit)
        hbox2.addWidget(self.source_btn)
        vbox.addLayout(hbox2)

        # 3. 실행 버튼
        self.run_btn = QPushButton('분류 및 라벨링 시작')
        self.run_btn.clicked.connect(self.start_classification)
        vbox.addWidget(self.run_btn)

        # 4. 진행률 바
        self.progress_bar = QProgressBar()
        vbox.addWidget(self.progress_bar)

        # 5. 로그 창
        self.log_edit = QPlainTextEdit()
        self.log_edit.setReadOnly(True)
        vbox.addWidget(self.log_edit)

        self.setGeometry(300, 300, 600, 400)
        self.show()

    def select_source(self):
        if self.radio_folder.isChecked():
            path = QFileDialog.getExistingDirectory(self, '이미지 폴더 선택')
        else: # self.radio_file.isChecked()
            path, _ = QFileDialog.getOpenFileName(self, '이미지 파일 선택', '', 'Image Files (*.png *.jpg *.jpeg *.bmp *.gif)')
        
        if path:
            self.source_path_edit.setText(path)

    def start_classification(self):
        model_path = MODEL_PATH
        source_path = self.source_path_edit.text()

        if not source_path:
            self.log_edit.appendPlainText("오류: '입력 경로'를 선택해주세요.")
            return

        # 타임스탬프 생성
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join('runs/detect', f'results({timestamp})')

        self.run_btn.setEnabled(False)
        self.progress_bar.setValue(0)
        self.log_edit.clear()

        self.worker = Worker(model_path, source_path, output_path)
        self.worker.progress.connect(self.progress_bar.setValue)
        self.worker.log.connect(self.log_edit.appendPlainText)
        self.worker.finished.connect(self.on_finished)
        self.worker.start()

    def on_finished(self, result_path):
        self.run_btn.setEnabled(True)
        self.worker = None # 다음 실행을 위해 worker 참조를 초기화

        # 전달받은 경로가 비어있지 않은 경우 (오류 없이 정상 종료된 경우) 메시지 표시
        if result_path:
            self.log_edit.appendPlainText(f"\n결과가 '{result_path}' 폴더에 저장되었습니다.")

    def closeEvent(self, event):
        if self.worker is not None and self.worker.isRunning():
            print("Closing... Stopping worker thread. Please wait.") # 터미널에 메시지 출력
            self.log_edit.appendPlainText("프로그램 종료 중... 작업 스레드가 멈추기를 기다립니다.")
            self.worker.stop()
            self.worker.wait() # 스레드가 완전히 끝날 때까지 기다림
            print("Worker thread finished. Exiting.")
        event.accept()

if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = ClassifierApp()
    sys.exit(app.exec_())

C:\Users\Lee\AppData\Local\Temp\ipykernel_21784\471447738.py:104: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Center' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[image_id, 'Label'] = predicted_class_name


SystemExit: 0

C:\Users\Lee\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [1]:
# ENG ver
# YOLO 모델로 이미지 분류 후 유형별로 폴더 자동 정리 PyQt 버전 + csv 라벨 자동 기입
import sys
import os
import shutil
import glob
import pandas as pd
from PyQt5.QtWidgets import (QApplication, QWidget, QVBoxLayout, QHBoxLayout, QGroupBox,
                             QPushButton, QLineEdit, QFileDialog, QLabel, 
                             QProgressBar, QPlainTextEdit, QRadioButton)
from PyQt5.QtCore import QThread, pyqtSignal
from ultralytics import YOLO
from datetime import datetime

# 모델 경로
MODEL_PATH = 'runs/detect/train/weights/best.pt'

# 긴 작업(YOLO 예측)을 처리할 별도의 스레드
class Worker(QThread):
    # 시그널 정의: 진행률 업데이트, 로그 메시지, 작업 완료
    progress = pyqtSignal(int)
    log = pyqtSignal(str)
    finished = pyqtSignal(str)

    def __init__(self, model_path, source_path, output_path):
        super().__init__()
        self.model_path = model_path
        self.source_path = source_path
        self.output_path = output_path
        self.is_running = True

    def run(self):
        try:
            # 1. 모델 로드
            if not os.path.exists(self.model_path):
                self.log.emit(f"Error: Model file not found. Please check the path: {self.model_path}")
                self.finished.emit("")
                return
            
            self.log.emit(f"'{os.path.basename(self.model_path)}' model Loading ...")
            model = YOLO(self.model_path)
            self.log.emit(f"Model loaded successfully. Classes: {list(model.names.values())}")

            # 2. CSV 파일 자동 탐지 및 로드
            csv_path_to_process = None
            df = None
            source_dir = self.source_path if os.path.isdir(self.source_path) else os.path.dirname(self.source_path)
            
            found_csv_files = glob.glob(os.path.join(source_dir, '*.csv'))
            if len(found_csv_files) == 1:
                csv_path_to_process = found_csv_files[0]
                df = pd.read_csv(csv_path_to_process)
                if 'ID' in df.columns:
                    df.set_index('ID', inplace=True)
                    self.log.emit(f"Loading CSV file '{os.path.basename(csv_path_to_process)}' to prepare for labeling.")
                else:
                    df = None
                    self.log.emit(f"Warning: 'ID' column not found in'{os.path.basename(csv_path_to_process)}' file. Skipping CSV labeling.")
            elif len(found_csv_files) > 1:
                self.log.emit(f"Warning: Multiple CSV files ({len(found_csv_files)}) found in the source folder. Skipping CSV labeling.")
            else:
                self.log.emit("Info: No CSV file found in the source folder. Skipping CSV labeling.")

            # 3. 이미지 파일 목록 가져오기
            if os.path.isdir(self.source_path):
                image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp', '*.gif']
                image_files = []
                for ext in image_extensions:
                    image_files.extend(glob.glob(os.path.join(self.source_path, ext)))
                if not image_files:
                    self.log.emit(f"Error: No images found in the '{self.source_path}' folder.")
                    self.finished.emit("")
                    return
            
            elif os.path.isfile(self.source_path):
                image_files = [self.source_path]

            else:
                self.log.emit(f"Error: Invalid path: '{self.source_path}'")
                self.finished.emit()
                return
            
            self.log.emit(f"Starting classification for a total of {len(image_files)} images.")
            os.makedirs(self.output_path, exist_ok=True)

            # 4. 이미지 분류, 파일 복사, CSV 업데이트 루프
            for i, image_path in enumerate(image_files):
                if not self.is_running:
                    break
                
                results = model.predict(source=image_path, verbose=False)
                result = results[0]
                image_filename = os.path.basename(image_path)
                image_id = int(os.path.splitext(image_filename)[0])

                if result.boxes:
                    top_class_id = int(result.boxes.cls[0])
                    predicted_class_name = model.names[top_class_id]
                else:
                    predicted_class_name = "Unclassified"

                # df가 있을 경우에만 라벨링 수행
                if df is not None:
                    if image_id in df.index:
                        df.loc[image_id, 'Label'] = predicted_class_name
                    else:
                        self.log.emit(f"Warning: ID {image_id} not found in the CSV file. Cannot write label.")

                # 파일 복사
                destination_folder = os.path.join(self.output_path, predicted_class_name)
                os.makedirs(destination_folder, exist_ok=True)
                shutil.copy(image_path, destination_folder)
                
                self.log.emit(f"'{image_filename}' -> '{predicted_class_name}'")
                self.progress.emit(int((i + 1) / len(image_files) * 100))

            # 5. 최종 CSV 파일 저장
            if df is not None:
                try:
                    df.reset_index(inplace=True)
                    output_csv_name = f"{os.path.splitext(os.path.basename(csv_path_to_process))[0]}_labeled.csv"
                    output_csv_path = os.path.join(self.output_path, output_csv_name)
                    df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
                    self.log.emit(f"\nLabeled CSV file saved: '{output_csv_path}'")
                except Exception as e:
                    self.log.emit(f"\nError occurred while saving CSV file: {e}")

            if self.is_running:
                self.log.emit("\nAll tasks have been completed!")
            else:
                self.log.emit("\nOperation was cancelled.")
        except Exception as e:
            self.log.emit(f"Error Occurred: {e}")
        finally:
            self.finished.emit(self.output_path)

    def stop(self):
        self.is_running = False

# 메인 윈도우
class ClassifierApp(QWidget):
    def __init__(self):
        super().__init__()
        self.initUI()
        self.worker = None

    def initUI(self):
        self.setWindowTitle('Wafer Map Classifier')
        
        # 레이아웃 설정
        vbox = QVBoxLayout()
        self.setLayout(vbox)

        # 1. 폴더/파일 선택 옵션 UI 추가 ---
        group_box = QGroupBox("Select Image Folder/File")
        self.radio_layout = QHBoxLayout()
        self.radio_folder = QRadioButton("Folder")
        self.radio_file = QRadioButton("File")
        self.radio_folder.setChecked(True) # 기본값으로 폴더 선택
        self.radio_layout.addWidget(self.radio_folder)
        self.radio_layout.addWidget(self.radio_file)
        group_box.setLayout(self.radio_layout)
        vbox.addWidget(group_box)

        # 2. 소스 폴더 선택
        hbox2 = QHBoxLayout()
        self.source_label = QLabel('Input Path:')
        self.source_path_edit = QLineEdit()
        self.source_path_edit.setReadOnly(True)
        self.source_btn = QPushButton('Select Path...')
        self.source_btn.clicked.connect(self.select_source)
        hbox2.addWidget(self.source_label)
        hbox2.addWidget(self.source_path_edit)
        hbox2.addWidget(self.source_btn)
        vbox.addLayout(hbox2)

        # 3. 실행 버튼
        self.run_btn = QPushButton('Start Classification & Labeling')
        self.run_btn.clicked.connect(self.start_classification)
        vbox.addWidget(self.run_btn)

        # 4. 진행률 바
        self.progress_bar = QProgressBar()
        vbox.addWidget(self.progress_bar)

        # 5. 로그 창
        self.log_edit = QPlainTextEdit()
        self.log_edit.setReadOnly(True)
        vbox.addWidget(self.log_edit)

        self.setGeometry(300, 300, 600, 400)
        self.show()

    def select_source(self):
        if self.radio_folder.isChecked():
            path = QFileDialog.getExistingDirectory(self, 'Select Image Folder')
        else: # self.radio_file.isChecked()
            path, _ = QFileDialog.getOpenFileName(self, 'Select Image File', '', 'Image Files (*.png *.jpg *.jpeg *.bmp *.gif)')
        
        if path:
            self.source_path_edit.setText(path)

    def start_classification(self):
        model_path = MODEL_PATH
        source_path = self.source_path_edit.text()

        if not source_path:
            self.log_edit.appendPlainText("Error: Please select an 'Input Path'.")
            return

        # 타임스탬프 생성
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join('runs/detect', f'results({timestamp})')

        self.run_btn.setEnabled(False)
        self.progress_bar.setValue(0)
        self.log_edit.clear()

        self.worker = Worker(model_path, source_path, output_path)
        self.worker.progress.connect(self.progress_bar.setValue)
        self.worker.log.connect(self.log_edit.appendPlainText)
        self.worker.finished.connect(self.on_finished)
        self.worker.start()

    def on_finished(self, result_path):
        self.run_btn.setEnabled(True)
        self.worker = None # 다음 실행을 위해 worker 참조를 초기화

        # 전달받은 경로가 비어있지 않은 경우 (오류 없이 정상 종료된 경우) 메시지 표시
        if result_path:
            self.log_edit.appendPlainText(f"\nResults have been saved to the '{result_path}' folder.")

    def closeEvent(self, event):
        if self.worker is not None and self.worker.isRunning():
            print("Closing... Stopping worker thread. Please wait.") # 터미널에 메시지 출력
            self.log_edit.appendPlainText("Closing program... Waiting for the worker thread to stop.")
            self.worker.stop()
            self.worker.wait() # 스레드가 완전히 끝날 때까지 기다림
            print("Worker thread finished. Exiting.")
        event.accept()

if __name__ == '__main__':
    app = QApplication(sys.argv)
    ex = ClassifierApp()
    sys.exit(app.exec_())

SystemExit: 0

C:\Users\Lee\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
